In [ ]:
from itertools import product
from collections import Counter
import os
import re
import pickle as pkl

from catboost import CatBoostClassifier
import nltk
from nltk.corpus import stopwords
import nltk.stem as st
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import Pipeline

from prepare_text_class import TextPrepareClass
from utilities import seedEverything

In [ ]:
rand_seed = 141592
seedEverything(rand_seed)

## Задаю переменные

In [ ]:
PATH_DATA = os.path.join('.', 'data')
PATH_SUBM = os.path.join('.', 'submissions')
PATH_MODL = os.path.join('.', 'models')

In [ ]:
PHONES = r'\b(samsung|galaxy|xiaomi|iphon|redmi|note|honor|huawei|apple|' +\
          'nokia|meizu|google|самсунг|айфон|mi|lenovo|lg|redme|asus|vivo|' +\
          'zte|helio|mediatek|oppo|htc|pixel|xperia|fly|realme|zenfone|' +\
          'alcatel|blade|philips|touch|lumia|oneplus|motorola|inoi|red|neo|' +\
          'moto|panasonic|band|honnor|bbk|vertex|lafleur|xiomi|редми|хонор|' +\
          'ноки|хуаве|мейзу|асус|галакси|иной|гэлакси|хонор)(|[a-zа-я]+)\b'

## Загрузка, очистка и преобразование данных

In [ ]:
df = pd.read_csv(os.path.join(PATH_DATA, 'ru_train.csv'), )
df.shape

In [ ]:
%%time
textprepare = TextPrepareClass(PHONES, stopwords.words('russian'))
df['text_cl'] = df.review.map(textprepare.clean_all)
df['target']  = df.rating.apply(lambda x: 0 if int(x) >= 4 else 1)

df = df.sample(frac=1).reset_index(drop=True)
df.sample(3)[['review', 'rating', 'link', 'target', 'text_cl']]

In [ ]:
df['target'].value_counts(normalize=True)

## Создание и сохранение модели и токенайзера для инференса через   
## onnx + docker + flask

Очищенные отзывы векторизую через tf-idf.   
Полученные векторы в CatBoostClassifier для подбора параметров.

In [ ]:
grid_pipe = Pipeline([('vector', TfidfVectorizer(analyzer = 'char_wb')),
                     ('model', CatBoostClassifier(verbose=400)),
                     ])

params_pipe = {'vector__max_features': [1024],       # 256, 512, 768, 
               'vector__ngram_range': [(2, 3), (3, 4),],
               #'vector__min_df': [0.2, 0.3],              # 0.5
               #'vector__max_df': [0.75, 0.8],             # 0.7, 1.0
               'model__iterations': [500, 1000],
               'model__learning_rate': [.03, .06],
               'model__depth': [3, 6],
               'model__l2_leaf_reg': [2, 3],
               
               }

In [ ]:
%%time
clf = GridSearchCV(grid_pipe, params_pipe,
                   n_jobs=-1, cv=5,
                   scoring = ['roc_auc', 'accuracy'],
                   refit='roc_auc',
                   verbose=0,
                  )
clf.fit(df.text_cl, df.target)
clf.best_score_, clf.best_params_

 Сохраняю результаты на всех возможных комбинациях заданных параметров   
 для дальнейшего анализа

In [ ]:
res = pd.DataFrame(clf.cv_results_['params'])
res['mean_test_roc_auc'] = clf.cv_results_['mean_test_roc_auc']
res['mean_test_accuracy'] = clf.cv_results_['mean_test_accuracy']
res['mean_sum'] = (clf.cv_results_['mean_test_accuracy'] +\
                   clf.cv_results_['mean_test_roc_auc']
                   )/2

res.to_csv(os.path.join(PATH_DATA, 'reslt_grid1_catboost.csv'), index=False)

## Обучаю модель на лучших параметрах и полных данных

In [ ]:
%%time
vectorizer = TfidfVectorizer(analyzer = 'char_wb',
                             ngram_range=clf.best_params_['vector__ngram_range'],
                             max_df=clf.best_params_['vector__max_df'],
                             min_df=clf.best_params_['vector__min_df'],
                             max_features=clf.best_params_['vector__max_features'],
                            )
vectorizer.fit(df.text_cl)
train = vectorizer.transform(df.text_cl)

model = (penalty=clf.best_params_['model__penalty'],
                           solver='liblinear',
                           C=clf.best_params_['model__C'],
                           class_weight=clf.best_params_['model__class_weight'],
                           max_iter=clf.best_params_['model__max_iter'],
                          )
model.fit(train, df.target)

## Сохраняю

С учетом предполагаемого применения, будет необходимо отслеживать версии    
скриптов очистки, приведения к начальной форме и модели. Для облегчения   
этолй задачи объеденим все в класс. При изменении версий данный класс не   
будет занимать много места, об этом не беспокоюсь.    

In [ ]:
with open(os.path.join(PATH_MODL, 'catboost_model.pkl'), 'wb') as fd:
    pkl.dump(model, fd)
    
with open(os.path.join(PATH_MODL, 'catboost_vektorizer.pkl'), 'wb') as fd:
    pkl.dump(vectorizer, fd)

## Посмотрим на результат обучения на полном датасете

In [ ]:
pred_train_cb = model.predict(train)
print( f'roc_auc_score: {roc_auc_score(df.target, pred_train_cb)}\n'\
       f'accuracy_score: {accuracy_score(df.target, pred_train_cb)}')

In [ ]:
cm = confusion_matrix(df.target, pred_train_cb)
cm

## Ошибки в предсказаниях тональности обзоров

In [ ]:
df['pred_train_cb'] = pred_train_cb
error_reviews = df[df['target'] != df['pred_train_cb']]

In [ ]:
tmp = error_reviews.sample()
print(f'True class {tmp.target.item()},\nPredicted class {tmp.pred_train_cb.item()}\n')
print(tmp.review.item())